In [567]:
import regex as re
#lifting pretokenizer regex from tiktoken
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""" 

In [568]:
text = """The quick brown fox jumps over the lazy dog while the cat watches from the windowsill. 
She doesn't understand why he'd rather run through the forest than stay home. 
I've seen them before in the mountains, they're quite remarkable creatures. 
We'll need to gather supplies: food, water, and rope. It's going to be an adventure! 
The weather forecast says it'll rain tomorrow, so we're preparing accordingly. 
They've already left the house without telling anyone where they're going. 
I'm not sure if we can't find another way around this problem. 
He's got everything he needs: maps, compass, and a good attitude."""

re.findall(PAT,text)[:6]

['The', ' quick', ' brown', ' fox', ' jumps', ' over']

In [569]:
def process_chunks(path, chunksize = 2**16, special_token = "<|endoftext|>"):
    # reads a text file path points to and creates chunks of size chunksize - separated by special token 
    # returns an iterable of chunks of bytes
    delimiter = special_token.encode("utf-8")
    # read bytes from the file
    with open(path,"rb") as f:
        buffer = b""
        while True:
            # read chunksize many bytes
            chunk = f.read(chunksize)
            if not chunk:
                # EOF - flush the buffer and exit the loop.
                yield buffer
                return
            # data after the last delimiter from previous chunk is saved in the buffer
            buffer += chunk
            # find the integer location of the latest delimiter
            id = buffer.rfind(delimiter)
            # if we can't find a delimiter - we keep going until we do.
            # this is in principle dangerous in case we dont'get a delimiter for very long.
            if id == -1:
                continue
            else:
                # if a delimiter is found, split it before the latest one 
                delimited = buffer[:id]
                # save the data after that and prepend it to the next chunk.
                buffer = buffer[id:]
                yield delimited


In [570]:
# UTF-8 encoding converts any character in the unicode to a prefix code of array of bytes 
b = "😊".encode("utf-8")
[bin(by) for by in list(b)], list(b)

(['0b11110000', '0b10011111', '0b10011000', '0b10001010'],
 [240, 159, 152, 138])

In [571]:
path = "data/test.txt"
special_tokens = ["<|endoftext|>", "<|padding|>", "<|mask|>"]
pattern = "|".join(special_tokens)
pattern = "|".join(re.escape(tok) for tok in special_tokens)

freq = {} # pretoken frequency map - find set of all unique pretokens and their frequencies
for chunk in process_chunks(path):
    for token in re.findall(PAT, re.sub(pattern," ",chunk.decode('utf-8'))):
        freq[token] = freq.get(token,0) + 1

dict(list(freq.items())[:10])

{'LOUISVILLE': 1,
 ',': 829,
 ' Ky': 1,
 '.': 800,
 ' —': 26,
 ' A': 33,
 ' few': 11,
 ' unflattering': 1,
 ' reviews': 1,
 ' are': 81}

every pretoken will be represented in terms of byte sequences of integers from 0 to 255 using utf-8 encoding. 

Then we will go through all adjacent pairs over all pretokens and find the most frequent pair. This will become a new token, and pretokens will be updated if they include this new pair, assigned a new integer as the token index in the sequence starting from 256. Repeat this until we reach the desired number of tokens.

we use a max heap to keep track of the most frequent pair using the frequency as a key for ordering. If a pair disappears in the process of merges, it is not removed from the heap to preserve the heap property. We simply consult the frequency map, which is the source of truth, every time we need the highest frequency pair. This is called lazy deletion. 

In [572]:
import heapq

pretoken_str, pretoken_freq = zip(*freq.items())
pretokens = [list(pretoken.encode("utf-8")) for pretoken in pretoken_str]

# once we have the frequency map - let's fix an index for each pretoken

pair_map = {}
for index in range(len(pretokens)):    
    token = pretokens[index]
    for i in range(len(token)-1):
        pair = (token[i],token[i+1])
        pair_map[pair] = pair_map.get(pair,0) + pretoken_freq[index]
        
# create a priority queue to keep track of the most frequent pair
pair_heap = [(-count,pair) for pair, count in pair_map.items()]
heapq.heapify(pair_heap)

# LAZY DELETION - pair_map is the source of truth
# pair heap might have stale entries as pairs get deleted
# instead of finding and deleting every single pair, we keep them and 
# check if they are still alive by comparing the most frequent pair
# to the pair_map, which is the source of truth. The point of not
# deleting from the heap is to maintain the heap property.

def mergebpairs(pretoken: list,pair: tuple, new_index: int):
    # given a pair we want to merge, it returns: 
    #   the pretoken with merges applied, 
    #   pairs to be added, 
    #   pairs to be removed from the frequency map
    if not pair:
        return pretoken 
    else:
        to_add = []
        to_remove = []
        i = 0
        while i < len(pretoken):
            if i < len(pretoken) - 1 and pair[0] == pretoken[i] and pair[1]  == pretoken[i+1]:
                left = []
                right = []
                if i > 0: #has left
                    left = pretoken[:i]
                    to_remove.append((pretoken[i-1],pretoken[i]))
                    to_add.append((pretoken[i-1],new_index))
                if i + 2 < len(pretoken): #has right
                    right = pretoken[i+2:]
                    to_remove.append((pretoken[i+1],pretoken[i+2]))
                    to_add.append((new_index,pretoken[i+2]))
                to_remove.append(pair)
                pretoken = left + [new_index] + right
            i +=1
        return pretoken, to_add, to_remove

def heap_entry_valid(heap_entry,pair_map):
    # check if the top node in the heap is stale
    freq = -heap_entry[0]
    pair = heap_entry[1]
    return pair in pair_map and pair_map[pair] == freq

def resolve(index):
    # given a token label - resolves it to an array from the base vocabulary
    if index < 256 or index not in merges:
        return [index]
    else:
        return [item for sublist in [resolve(children) for children in merges[index]] for item in sublist]

def decode(tokens):
    # breaks down a list of tokens into base representation 
    return [bt for resolved in [resolve(index) for index in tokens] for bt in resolved]



In [573]:
number_of_tokens = 3000 #has to be larger than 256 
merges = {}
k = number_of_tokens - 256
for merge_index in range(k):
    if len(pair_heap) == 0:
        break
    
    # LAZY DELETE: ensure top of the heap is valid((pair_heap[0][1] in pair_map)
    while not heap_entry_valid(pair_heap[0],pair_map):
        heapq.heappop(pair_heap)

    # find the highest frequency entry that is lexographically largest
    degenerate_stack = []
    highest_freq = pair_heap[0][0]
    while pair_heap[0][0] == highest_freq:
        val = heapq.heappop(pair_heap)
        if heap_entry_valid(val,pair_map):
            degenerate_stack.append(val)
    
    #lex_sort = sorted(degenerate_stack, key=lambda x: x[1])
    # lexographically largest is ill defined because the tokens may not correspond to 
    # valid unicode characters because the byte sequence is not necessarily utf decodable (could be incomplete)
    # instead of decoding to utf-8 and comparing, we compare the byte sequences instead. 
    lex_sort = sorted(degenerate_stack, key=lambda heap_entry: tuple(bytes(decode([index])) for index in heap_entry[1]))
    most_frequent_pair = lex_sort[-1] #grab the largest
    new_pair_index = 256 + merge_index
    merges[new_pair_index] = most_frequent_pair[1]
    # push rest back into the heap
    for pair in lex_sort[:-1]:
        heapq.heappush(pair_heap,pair)
    
    # print ('most frequent pair ', most_frequent_pair[1],':', pair_map[most_frequent_pair[1]])
    pair_map_additive_diff = {}
    for i in range(len(pretokens)):
        pretokens[i], created, destroyed = mergebpairs(pretokens[i],most_frequent_pair[1],new_pair_index)
        for pair in created:
            pair_map_additive_diff[pair] = pair_map_additive_diff.get(pair,0) + pretoken_freq[i]
        # remove the destroyed pairs from pair_map 
        for pair in destroyed:
            if pair in pair_map:
                pair_map[pair] -= pretoken_freq[i]
            if pair in pair_map and pair_map[pair] == 0:
                del pair_map[pair]

    for new_pair in pair_map_additive_diff:
        updated_frequency = pair_map.get(new_pair,0) + pair_map_additive_diff[new_pair]
        pair_map[new_pair] = updated_frequency
        heapq.heappush(pair_heap,(-updated_frequency,new_pair))



the dictionary of merges tells us how each new token we created is made of

In [574]:
dict(list(merges.items())[:10]) #just showing the first 10 merges

{256: (32, 116),
 257: (104, 101),
 258: (32, 97),
 259: (105, 110),
 260: (114, 101),
 261: (256, 257),
 262: (32, 115),
 263: (111, 110),
 264: (32, 119),
 265: (32, 99)}

In [575]:
for pretoken in pretokens:
    print ('-'.join([bytes(resolve(index)).decode('utf-8', errors='replace') for index in pretoken]))

L-OU-IS-V-I-L-L-E
,
 K-y
.
 —
 A
 few
 unf-la-ttering
 re-view-s
 are
 t-o
 be
 expected
 with
 any
 hotel
 particula-rly
 one
 whose
 -rates
 sta-rt
 at
 $
4-9
 per
 night
 But
 w-h-i-le
 comp-la-in-t-s
 about
 sha-bby
 rooms
 and
 t-h-in
 t-ow-e-ls
 c-ommon
 in
 the
 ind-u-s-t-ry
 one-s
 like
 these
 f-r-o-m
 T-rip-A-d-vis-o-r
c-o-m
 n-ot
:
 “
I-t
 is
 a
 clean
 but
 there
 -l-ot
 of
 homeless
 people
.”
R-u-n
 far
 away
!!-!!-!
 This
 shelter
!-”
D-O
 N-OT
 S-T-AY
 H-E-R-E
 U-N-L-E-S-S
 Y-OU
 A-R-E
 H-OM-E-L-E-S-S
…
 A-l-l
 w-o-r-ke-rs
 former
 addict-s
/
homeless
 Hotel
 Louisville
 1-2
 st-o-r-ies
 b-r-ic-k
 ad-o-rned
 large
 white
 c-ross
 in-de-e-d
 event
 spa-c-e
 open
 public
 At
 same
 t-i-me
 it
 trans-itional
-
housing
 facility
 substance
a-b-u-s-e
 recovery
 cent-e-r
 job
training
 site
 owne-d
 ope-rated
 by
 Wayside
 Ch-rist-ian
 M-is-s-ion
 nonprofit
 that
 shelte-rs
 fee-ds
 city
’
s
 p-opulation
 b-ought
 building
 fore-c-l-os-u-re
 au-c-t-ion
 200-9
 never
 inten-di

In [ ]:
c = '桁'.encode('utf-8')
230.to_bytes(2,)

SyntaxError: invalid decimal literal (1313855075.py, line 2)

In [274]:
b"\xf0\x9f\x98\x8a".decode('utf-8'), list(b"\xf0\x9f\x98\x8a")

('😊', [240, 159, 152, 138])

[240, 159, 152, 138]

In [183]:
list('abcd*@桁'.encode('utf8'))

[97, 98, 99, 100, 42, 64, 230, 161, 129]

In [213]:
b"".join([i.to_bytes(1,'big') for i in [64, 230, 161, 129]]).decode('utf-8')

'@桁'

In [217]:
bytes([65,2,3,64, 230, 161, 129]).decode('utf-8')

'A\x02\x03@桁'